#      Atelier Scikit-learn

-   Contexte

Une entreprise possède plusieurs bâtiments équipés de capteurs IoT. 
Chaque capteur collecte régulièrement des informations sur la température, l'humidité, la pression, 
la consommation énergétique, le bâtiment, la date et l'heure de la mesure.  
Chaque mesure possède également un état (OK, ALERTE et ERREUR). 
L'objectif de l'atelier est de construire un modèle capable de prédire automatiquement l'état d'un 
capteur à partir de ses mesures. 
L'atelier suivra le workflow classique du Machine Learning : 
Dataset → Chargement → Exploration → Nettoyage → X / y → Train / Test → Prétraitement → 
Modèle → fit()→ predict()→ Évaluation → Sauvegarde → Chargement → Réutilisation

# Partie 0 – mise en place de l’environnement

2) Stocker le fichier fourni mesures_capteurs.csv dans le sous-dossier data 
3) Créer le notebook atelier_scikit-learn_iot.ipynb 
4) Installer et importer seaborn, matplotlib et pandas 


In [23]:
%pip install pandas numpy matplotlib seaborn scikit-learn joblib

Note: you may need to restart the kernel to use updated packages.


- Importation des bibliothèques

In [24]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

- Import des outils de Scikit-learn

In [25]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score,confusion_matrix,classification_report)

In [26]:
import joblib
import pickle

df =[]

5) Importer mesures_capteurs.csv dans le dataframe df 

In [27]:
df = pd.read_csv("../data/mesures_capteurs.csv")

6) Explorer le dataframe df

In [28]:
df.head() # Affichage des premieres lignes

,id_mesure,date_heure,id_capteur,batiment,temperature,humidite,pression,consommation,etat
0,M0413,2026-01-22 04:00:00,C005,B002,25.46,58.06,1008.95,287.28,OK
1,M0290,2026-01-17 01:00:00,C002,B001,24.00,79.73,993.39,116.20,OK
2,M0077,2026-01-08 04:00:00,C005,B002,25.82,54.47,1010.32,288.50,OK
3,M0079,2026-01-08 06:00:00,C007,B003,28.23,69.39,1019.62,136.65,OK
4,M0183,2026-01-12 14:00:00,C003,B001,20.58,53.80,1016.58,182.62,OK


In [29]:
df.shape 

(605, 9)

In [30]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 605 entries, 0 to 604
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id_mesure     605 non-null    object 
 1   date_heure    605 non-null    object 
 2   id_capteur    605 non-null    object 
 3   batiment      605 non-null    object 
 4   temperature   599 non-null    float64
 5   humidite      600 non-null    float64
 6   pression      600 non-null    float64
 7   consommation  600 non-null    float64
 8   etat          601 non-null    object 
dtypes: float64(4), object(5)
memory usage: 42.7+ KB


In [31]:
df.describe()


,temperature,humidite,pression,consommation
count,599.000000,600.00000,600.000000,600.000000
mean,24.878314,64.92620,1012.221900,208.675417
std,4.059576,10.76905,10.599042,72.243567
min,-18.500000,28.52000,850.000000,18.120000
25%,22.570000,58.17250,1006.790000,160.177500
50%,24.860000,65.37500,1012.855000,206.150000
75%,27.275000,71.61500,1017.827500,254.127500
max,58.700000,145.00000,1038.430000,875.000000


# Partie 1 – Gestion des doublons

Avec Pandas, 
1) vérifier l’existence de doublons dans df

In [32]:
df.duplicated().sum()

np.int64(5)

Pour afficher les doublons :

In [33]:
df[df.duplicated()]

,id_mesure,date_heure,id_capteur,batiment,temperature,humidite,pression,consommation,etat
183,M0599,2026-01-29 22:00:00,C011,B004,22.02,68.09,1005.90,227.81,OK
231,M0026,2026-01-06 01:00:00,C002,B001,22.87,77.99,1010.15,213.19,OK
355,M0147,2026-01-11 02:00:00,C003,B001,26.14,84.97,1003.59,142.31,OK
538,M0456,2026-01-23 23:00:00,C012,B004,20.68,72.69,1022.77,309.01,OK
539,M0302,2026-01-17 13:00:00,C002,B001,22.05,58.26,1007.30,140.42,OK


2) le cas échéant, supprimer les doublons puis vérifier la suppression  

In [34]:
df = df.drop_duplicates()

In [35]:
df.duplicated().sum()

np.int64(0)

# Partie 2 – Sélection de y (cible) et X (caractéristiques) 

1) Définir "etat" comme la cible ou valeur à prédire et "temperature", "humidite", 
"pression" et "consommation" comme caractéristiques ou variables explicatives

In [36]:
# Definissons x contenant les variables explicatives
x = df[
    [
        "temperature",
        "humidite",
        "pression",
        "consommation"
    ]
]

In [37]:
# Definissons y la variable a predire
y = df["etat"]

In [38]:
x.head(5)

,temperature,humidite,pression,consommation
0,25.46,58.06,1008.95,287.28
1,24.00,79.73,993.39,116.20
2,25.82,54.47,1010.32,288.50
3,28.23,69.39,1019.62,136.65
4,20.58,53.80,1016.58,182.62


In [39]:
y.head(5)

0    OK
1    OK
2    OK
3    OK
4    OK
Name: etat, dtype: object

3) Quel est le type du problème de machine learning ?  

Il s'agit d'un probleme de Classification supervisée multiclasse. 

# Partie 3 – Découpage Train/Test

Diviser X en deux ensembles distincts : un pour l'entraînement (train) et un pour le test (test). 
Avec les conditions suivantes : 20% des données serviront au test ; garantir la reproductibilité du 
découpage ; conserver les mêmes proportions de classes dans l'ensemble de train et de test que 
dans les données d'origine. 

In [43]:
df = df.dropna(subset=["etat"])

In [44]:
print(df["etat"].isnull().sum())

0


In [45]:
x = df[
    [
        "temperature",
        "humidite",
        "pression",
        "consommation"
    ]
]

In [46]:
# Definissons y la variable a predire
y = df["etat"]

In [47]:
print(x.shape)
print(y.shape)

(596, 4)
(596,)


In [48]:
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)

In [49]:
y.value_counts(normalize=True)

etat
OK        0.942953
ALERTE    0.048658
ERREUR    0.008389
Name: proportion, dtype: float64

In [50]:
y_train.value_counts(normalize=True)

etat
OK        0.943277
ALERTE    0.048319
ERREUR    0.008403
Name: proportion, dtype: float64

In [51]:
y_test.value_counts(normalize=True)

etat
OK        0.941667
ALERTE    0.050000
ERREUR    0.008333
Name: proportion, dtype: float64

# Partie 4 – Gestion des valeurs manquantes

1) Vérifier l’existence de valeurs manquantes 

In [52]:
x.isnull().sum()

temperature     6
humidite        5
pression        5
consommation    5
dtype: int64

2) Sélectionner SimpleImputer avec la médiane 

In [53]:
imputer = SimpleImputer(
    strategy="median"
)

3) Qu’est ce qui justifie le choix de la médiane ?

La médiane est particulièrement utile lorsque les données contiennent des :
- valeurs extrêmes ;
- anomalies ;
- outliers.

Elle est moins influencée par les valeurs extrêmes que la moyenne.

4) Trouver les paramètres (médianes) de l’imputeur sur X_train

In [54]:
imputer.fit(x_train)

,missing_values,nan
,strategy,'median'
,fill_value,None
,copy,True
,add_indicator,False
,keep_empty_features,False


In [56]:
# Affichage des medianes calcules 
mediane_imputer = pd.Series(imputer.statistics_,index=x_train.columns)
print(mediane_imputer)

temperature       24.90
humidite          65.38
pression        1012.30
consommation     206.59
dtype: float64


5) Déterminer X_train_imputed et X_test_imputed, les transformés de X_train et X_test 

In [59]:
x_train_imputed = imputer.transform(x_train)  #Le transformé de x_train

In [60]:
x_test_imputed = imputer.transform(x_test)  # Le transformé de x_test